In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ------------------------------------------
# SETTINGS
STATE_LGAS = {
    "Katsina": [
        "Bakori", "Batagarawa", "Batsari", "Baure", "Bindawa", "Charanchi",
        "Dan Musa", "Dandume", "Danja", "Daura", "Dutsi", "Dutsin-Ma",
        "Faskari", "Funtua", "Ingawa", "Jibia", "Kafur", "Kaita", "Kankara",
        "Kankia", "Katsina", "Kurfi", "Kusada", "Mai'Adua", "Malumfashi",
        "Mani", "Mashi", "Matazu", "Musawa", "Rimi", "Sabuwa", "Safana",
        "Sandamu", "Zango"
    ],
    # Add more states and LGAs here as needed
}

WARD_BROWSE_URL = "https://integrity.ng/index.php/wards/browse"
INEC_PU_URL = "https://www.inecnigeria.org/polling-units/"
# ------------------------------------------

def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)

def fetch_wards(driver, state_name, lga_list):
    wards = []
    page = 0
    while True:
        driver.get(f"{WARD_BROWSE_URL}?page={page}")
        try:
            table = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "table"))
            )
        except:
            break  # No more pages

        rows = table.find_elements(By.TAG_NAME, "tr")[1:]
        if not rows:
            break

        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 4:
                lga = cols[2].text.strip()
                ward = cols[3].text.strip()
                if lga in lga_list:
                    wards.append({"LGA": lga, "Ward": ward})
        print(f"✅ Page {page}: {len(rows)} rows")
        page += 1
        time.sleep(1)
    return wards

def fetch_polling_units(driver, state_name, wards, lga_list):
    pus = []

    driver.get(INEC_PU_URL)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "selState")))

    try:
        state_select = Select(driver.find_element(By.ID, "selState"))
        state_select.select_by_visible_text(state_name)
        time.sleep(2)
    except:
        print(f"❌ State {state_name} not found in dropdown.")
        return pus

    for ward_data in wards:
        lga = ward_data['LGA']
        ward = ward_data['Ward']

        # Skip LGAs not in predefined list
        if lga not in lga_list:
            continue

        try:
            lga_select = Select(driver.find_element(By.ID, "selLga"))
            lga_select.select_by_visible_text(lga)
            time.sleep(1.5)

            ward_select = Select(driver.find_element(By.ID, "selWard"))
            ward_select.select_by_visible_text(ward)
            time.sleep(1)

            driver.find_element(By.ID, "btnSearch").click()
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "tblPu")))
            table = driver.find_element(By.ID, "tblPu")
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]

            for r in rows:
                cols = r.find_elements(By.TAG_NAME, "td")
                if len(cols) >= 1:
                    pus.append({
                        "LGA": lga,
                        "Ward": ward,
                        "Polling Station": cols[0].text.strip()
                    })

            print(f"   ✔️ {lga} / {ward}: {len(rows)} polling units")
        except:
            print(f"   ⚠️ Skipped: {lga} / {ward}")
        time.sleep(1.5)

    # Final strict filter to remove any polling units not in predefined LGAs/Wards
    pus = [pu for pu in pus if pu["LGA"] in lga_list and any(w['Ward'] == pu['Ward'] for w in wards)]
    
    return pus

def save_to_excel(state_name, wards, pus):
    df_wards = pd.DataFrame(wards).drop_duplicates()
    df_pus = pd.DataFrame(pus).drop_duplicates()
    df_lga = pd.DataFrame(sorted(df_wards["LGA"].unique()), columns=["LGA"])

    filename = f"{state_name}_LGA_Wards_PollingUnits.xlsx"
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name="LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="Wards", index=False)
        df_pus.to_excel(writer, sheet_name="PollingUnits", index=False)

    print(f"\n✅ Saved to {filename}\n")

def run_for_state(state_name, lga_list):
    print(f"\n🚀 Starting scrape for: {state_name}")
    driver = init_driver()
    wards = fetch_wards(driver, state_name, lga_list)
    print(f"📌 Total Wards Collected: {len(wards)}")

    pus = fetch_polling_units(driver, state_name, wards, lga_list)
    print(f"📌 Total Polling Units Collected: {len(pus)}")

    save_to_excel(state_name, wards, pus)
    driver.quit()

# Run for all states in STATE_LGAS
for state, lgas in STATE_LGAS.items():
    run_for_state(state, lgas)



🚀 Starting scrape for: Katsina
✅ Page 0: 100 rows
✅ Page 1: 100 rows
✅ Page 2: 100 rows
✅ Page 3: 100 rows
✅ Page 4: 100 rows
✅ Page 5: 100 rows
✅ Page 6: 100 rows
✅ Page 7: 100 rows
✅ Page 8: 100 rows
✅ Page 9: 100 rows
✅ Page 10: 100 rows
✅ Page 11: 100 rows
✅ Page 12: 100 rows
✅ Page 13: 100 rows
✅ Page 14: 100 rows
✅ Page 15: 100 rows
✅ Page 16: 100 rows
✅ Page 17: 100 rows
✅ Page 18: 100 rows
✅ Page 19: 100 rows
✅ Page 20: 100 rows
✅ Page 21: 100 rows
✅ Page 22: 100 rows
✅ Page 23: 100 rows
✅ Page 24: 100 rows
✅ Page 25: 100 rows
✅ Page 26: 100 rows
✅ Page 27: 100 rows
✅ Page 28: 100 rows
✅ Page 29: 100 rows
✅ Page 30: 100 rows
✅ Page 31: 100 rows
✅ Page 32: 100 rows
✅ Page 33: 100 rows
✅ Page 34: 100 rows
✅ Page 35: 100 rows
✅ Page 36: 100 rows
✅ Page 37: 100 rows
✅ Page 38: 100 rows
✅ Page 39: 100 rows
✅ Page 40: 100 rows
✅ Page 41: 100 rows
✅ Page 42: 100 rows
✅ Page 43: 100 rows
✅ Page 44: 100 rows
✅ Page 45: 100 rows
✅ Page 46: 100 rows
✅ Page 47: 100 rows
✅ Page 48: 100 row